# Notebook 5: Final Evaluation

**What this does (in simple terms):**

This is the exam day. We take both models (teacher and student) and
test them on audio they've NEVER seen before — the evaluation set
contains attack types A07-A19 that were not in the training data.

We measure:
- **EER**: Equal Error Rate — lower is better. The point where the rate
  of falsely accepting fakes equals the rate of falsely rejecting real audio.
- **min-tDCF**: A cost function used in the ASVspoof challenge.
- **Accuracy, Precision, Recall, F1**: Standard classification metrics.
- **Confusion Matrix**: Visual breakdown of correct/incorrect predictions.
- **Per-Attack EER**: How well the model handles each specific type of fake.

**Run on Colab with GPU for speed.**

## 5.1 Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers -q

In [ ]:
import os, json, random, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchaudio
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve)
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
PREPROCESSED_DIR = Path("/content/preprocessed")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/deepfake_project/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/deepfake_project/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR = 16000; MAX_DURATION = 6; MAX_SAMPLES = TARGET_SR * MAX_DURATION
BATCH_SIZE = 8; SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Verify everything exists
assert PREPROCESSED_DIR.exists(), "Run Notebook 1 first!"
assert (CHECKPOINT_DIR / "teacher_best.pt").exists(), "Run Notebook 3 first!"
assert (CHECKPOINT_DIR / "student_best.pt").exists(), "Run Notebook 4 first!"
print("All checkpoints found ✓")

## 5.2 Load Eval Data

In [ ]:
def load_protocol(path):
    df = pd.read_csv(path, sep=" ", header=None,
                     names=["speaker_id", "audio_id", "system_id", "attack_type", "label"])
    df["label_int"] = (df["label"] == "spoof").astype(int)
    return df

PROTOCOL_DIR = PREPROCESSED_DIR / "protocols"
eval_protocol = load_protocol(PROTOCOL_DIR / "ASVspoof2019.LA.cm.eval.trl.txt")
metadata = pd.read_csv(PREPROCESSED_DIR / "metadata.csv")

print(f"Eval samples: {len(eval_protocol)}")
print(f"Attack types: {sorted(eval_protocol['attack_type'].unique())}")

In [ ]:
class ASVspoofDataset(Dataset):
    def __init__(self, protocol_df, audio_dir, metadata_df):
        self.protocol = protocol_df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.length_lookup = {}
        for _, r in metadata_df.iterrows():
            if r.get("status") == "success":
                self.length_lookup[r["audio_id"]] = int(r["original_length"])
    def __len__(self): return len(self.protocol)
    def __getitem__(self, idx):
        row = self.protocol.iloc[idx]
        audio, _ = torchaudio.load(str(self.audio_dir / f"{row['audio_id']}.flac"))
        audio = audio.squeeze(0)
        mx = torch.max(torch.abs(audio))
        if mx > 0: audio = audio / mx
        ol = self.length_lookup.get(row["audio_id"], MAX_SAMPLES)
        mask = torch.zeros(MAX_SAMPLES); mask[:ol] = 1.0
        return {"audio": audio, "mask": mask,
                "label": torch.tensor(row["label_int"], dtype=torch.long),
                "audio_id": row["audio_id"], "attack_type": row["attack_type"]}

eval_dataset = ASVspoofDataset(eval_protocol, PREPROCESSED_DIR / "eval", metadata)
eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)
print(f"Eval: {len(eval_dataset)} samples, {len(eval_loader)} batches")

## 5.3 Model Architecture (for loading checkpoints)

In [ ]:
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.a_src = nn.Linear(out_dim, 1, bias=False)
        self.a_dst = nn.Linear(out_dim, 1, bias=False)
        self.leaky_relu = nn.LeakyReLU(0.2)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        h = self.W(x)
        attn = self.leaky_relu(self.a_src(h) + self.a_dst(h).transpose(-2,-1))
        return torch.bmm(self.dropout(F.softmax(attn, dim=-1)), h)

class MultiHeadGraphAttention(nn.Module):
    def __init__(self, in_dim, out_dim, n_heads=2, dropout=0.1):
        super().__init__()
        self.heads = nn.ModuleList([GraphAttentionLayer(in_dim, out_dim, dropout) for _ in range(n_heads)])
        self.norm = nn.LayerNorm(out_dim * n_heads)
    def forward(self, x):
        return self.norm(torch.cat([h(x) for h in self.heads], dim=-1))

class AASSISTBackend(nn.Module):
    def __init__(self, input_dim=1024, hidden_dim=160, n_heads=2, dropout=0.1):
        super().__init__()
        self.projection = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.GELU(),
                                         nn.LayerNorm(hidden_dim), nn.Dropout(dropout))
        self.spectral_gat = MultiHeadGraphAttention(hidden_dim, hidden_dim, n_heads, dropout)
        self.spectral_pool = nn.AdaptiveAvgPool1d(1)
        self.temporal_gat = MultiHeadGraphAttention(hidden_dim, hidden_dim, n_heads, dropout)
        self.temporal_pool = nn.AdaptiveAvgPool1d(1)
        cd = hidden_dim * n_heads * 2
        self.hetero_attention = nn.Sequential(nn.Linear(cd, hidden_dim), nn.GELU(),
                                               nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim))
        self.classifier = nn.Sequential(nn.Linear(hidden_dim, 64), nn.GELU(),
                                         nn.Dropout(dropout), nn.Linear(64, 2))
    def forward(self, f):
        x = self.projection(f)
        sp = self.spectral_pool(self.spectral_gat(x).transpose(1,2)).squeeze(-1)
        tp = self.temporal_pool(self.temporal_gat(x).transpose(1,2)).squeeze(-1)
        return self.classifier(self.hetero_attention(torch.cat([sp, tp], dim=-1)))

class AASSISTLBackend(nn.Module):
    def __init__(self, input_dim=1024, hidden_dim=64, n_heads=2, dropout=0.1):
        super().__init__()
        self.projection = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.GELU(),
                                         nn.LayerNorm(hidden_dim), nn.Dropout(dropout))
        self.spectral_gat = MultiHeadGraphAttention(hidden_dim, hidden_dim, n_heads, dropout)
        self.spectral_pool = nn.AdaptiveAvgPool1d(1)
        self.temporal_gat = MultiHeadGraphAttention(hidden_dim, hidden_dim, n_heads, dropout)
        self.temporal_pool = nn.AdaptiveAvgPool1d(1)
        cd = hidden_dim * n_heads * 2
        self.hetero_attention = nn.Sequential(nn.Linear(cd, hidden_dim), nn.GELU(),
                                               nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim))
        self.classifier = nn.Sequential(nn.Linear(hidden_dim, 32), nn.GELU(),
                                         nn.Dropout(dropout), nn.Linear(32, 2))
    def forward(self, f):
        x = self.projection(f)
        sp = self.spectral_pool(self.spectral_gat(x).transpose(1,2)).squeeze(-1)
        tp = self.temporal_pool(self.temporal_gat(x).transpose(1,2)).squeeze(-1)
        return self.classifier(self.hetero_attention(torch.cat([sp, tp], dim=-1)))

class TeacherModel(nn.Module):
    def __init__(self, w2v_name="facebook/wav2vec2-xls-r-300m", hidden_dim=160):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(w2v_name)
        self.wav2vec2.feature_extractor._freeze_parameters()
        self.backend = AASSISTBackend(input_dim=self.wav2vec2.config.hidden_size, hidden_dim=hidden_dim)
    def forward(self, audio, mask=None):
        if mask is not None: mask = mask.long()
        return self.backend(self.wav2vec2(audio, attention_mask=mask).last_hidden_state)

class StudentModel(nn.Module):
    def __init__(self, w2v_name="facebook/wav2vec2-xls-r-300m", hidden_dim=64):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(w2v_name)
        for p in self.wav2vec2.parameters(): p.requires_grad = False
        self.backend = AASSISTLBackend(input_dim=self.wav2vec2.config.hidden_size, hidden_dim=hidden_dim)
    def forward(self, audio, mask=None):
        if mask is not None: mask = mask.long()
        with torch.no_grad():
            features = self.wav2vec2(audio, attention_mask=mask).last_hidden_state
        return self.backend(features)

In [ ]:
def count_params(m, trainable=True):
    if trainable: return sum(p.numel() for p in m.parameters() if p.requires_grad)
    return sum(p.numel() for p in m.parameters())

print("Loading models...")
teacher = TeacherModel().to(DEVICE)
teacher.load_state_dict(torch.load(str(CHECKPOINT_DIR/"teacher_best.pt"), map_location=DEVICE)["model_state_dict"])
teacher.eval()
print(f"Teacher loaded. Trainable params: {count_params(teacher, True):,}")

student = StudentModel().to(DEVICE)
student.load_state_dict(torch.load(str(CHECKPOINT_DIR/"student_best.pt"), map_location=DEVICE)["model_state_dict"])
student.eval()
print(f"Student loaded. Trainable params: {count_params(student, True):,}")

## 5.4 Evaluation Metrics

In [ ]:
def compute_eer(labels, scores):
    fpr, tpr, th = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr; idx = np.nanargmin(np.abs(fpr - fnr))
    return (fpr[idx] + fnr[idx]) / 2, th[idx]

def compute_min_tdcf(labels, scores, Pspoof=0.05, Cmiss=1, Cfa=10):
    """
    Simplified min-tDCF (CM scores only).
    The full version requires ASV scores which are not available here.
    """
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    beta = Pspoof / (1 - Pspoof)
    return np.min(Cmiss * fnr + Cfa * beta * fpr)

@torch.no_grad()
def full_evaluation(model, dataloader, device, desc="Eval"):
    model.eval()
    all_scores, all_preds, all_labels = [], [], []
    all_ids, all_attacks = [], []

    for batch in tqdm(dataloader, desc=desc):
        audio = batch["audio"].to(device); mask = batch["mask"].to(device)
        labels = batch["label"].to(device)
        logits = model(audio, mask)
        probs = F.softmax(logits, dim=-1)
        all_scores.extend(probs[:, 1].cpu().numpy())
        all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_ids.extend(batch["audio_id"])
        all_attacks.extend(batch["attack_type"])

    labels = np.array(all_labels); scores = np.array(all_scores); preds = np.array(all_preds)
    eer, eer_th = compute_eer(labels, scores)
    return {
        "eer": eer, "eer_threshold": eer_th,
        "min_tdcf": compute_min_tdcf(labels, scores),
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "labels": labels, "scores": scores, "preds": preds,
        "attacks": np.array(all_attacks),
    }

## 5.5 Run Evaluation

In [ ]:
print("=" * 60)
print("EVALUATING ON EVAL SET (Unseen Attacks A07-A19)")
print("=" * 60)

print("\nTeacher evaluation...")
teacher_res = full_evaluation(teacher, eval_loader, DEVICE, "Teacher")

print("\nStudent evaluation...")
student_res = full_evaluation(student, eval_loader, DEVICE, "Student")

## 5.6 Results Comparison

In [ ]:
print("\n" + "=" * 60)
print("TEACHER vs STUDENT — EVAL SET RESULTS")
print("=" * 60)

rows = [
    ("EER (%)",     f"{teacher_res['eer']*100:.2f}%",    f"{student_res['eer']*100:.2f}%"),
    ("min-tDCF",    f"{teacher_res['min_tdcf']:.4f}",    f"{student_res['min_tdcf']:.4f}"),
    ("Accuracy",    f"{teacher_res['accuracy']:.4f}",     f"{student_res['accuracy']:.4f}"),
    ("Precision",   f"{teacher_res['precision']:.4f}",    f"{student_res['precision']:.4f}"),
    ("Recall",      f"{teacher_res['recall']:.4f}",       f"{student_res['recall']:.4f}"),
    ("F1-Score",    f"{teacher_res['f1']:.4f}",           f"{student_res['f1']:.4f}"),
    ("Params (trainable)", f"{count_params(teacher,True):,}", f"{count_params(student,True):,}"),
]

print(f"\n{'Metric':<20} {'Teacher':>14} {'Student':>14}")
print("-" * 50)
for name, t_val, s_val in rows:
    print(f"{name:<20} {t_val:>14} {s_val:>14}")

## 5.7 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, res, title in [(axes[0], teacher_res, "Teacher"), (axes[1], student_res, "Student")]:
    cm = confusion_matrix(res["labels"], res["preds"])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Bonafide", "Spoof"])
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title(f"{title}\nEER: {res['eer']*100:.2f}% | F1: {res['f1']:.4f}")

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "confusion_matrices.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5.8 Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, res, title in [(axes[0], teacher_res, "Teacher"), (axes[1], student_res, "Student")]:
    bon = res["scores"][res["labels"] == 0]
    spf = res["scores"][res["labels"] == 1]
    ax.hist(bon, bins=50, alpha=0.6, label="Bonafide", color="green", density=True)
    ax.hist(spf, bins=50, alpha=0.6, label="Spoof", color="red", density=True)
    ax.axvline(res["eer_threshold"], color="black", linestyle="--",
               label=f"EER threshold ({res['eer_threshold']:.3f})")
    ax.set_xlabel("P(spoof)"); ax.set_ylabel("Density")
    ax.set_title(f"{title}: Score Distribution"); ax.legend()

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "score_distributions.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5.9 Per-Attack EER Analysis

In [ ]:
def per_attack_eer(results):
    labels = results["labels"]; scores = results["scores"]; attacks = results["attacks"]
    rows = []
    for attack in sorted(set(attacks)):
        if attack == "-": continue
        mask = (attacks == attack) | (labels == 0)
        sub_l = labels[mask]; sub_s = scores[mask]
        if len(np.unique(sub_l)) < 2: continue
        eer, _ = compute_eer(sub_l, sub_s)
        rows.append({"Attack": attack, "Samples": int((attacks == attack).sum()),
                      "EER (%)": round(eer * 100, 2)})
    return pd.DataFrame(rows)

In [ ]:
print("\nPer-Attack EER — Teacher:")
t_attacks = per_attack_eer(teacher_res)
print(t_attacks.to_string(index=False))

print("\nPer-Attack EER — Student:")
s_attacks = per_attack_eer(student_res)
print(s_attacks.to_string(index=False))

In [ ]:
# Bar chart
if not t_attacks.empty and not s_attacks.empty:
    merged = t_attacks[["Attack","EER (%)"]].rename(columns={"EER (%)":"Teacher"}).merge(
        s_attacks[["Attack","EER (%)"]].rename(columns={"EER (%)":"Student"}), on="Attack")

    x = np.arange(len(merged)); w = 0.35
    fig, ax = plt.subplots(figsize=(16, 6))
    b1 = ax.bar(x - w/2, merged["Teacher"], w, label="Teacher", color="steelblue")
    b2 = ax.bar(x + w/2, merged["Student"], w, label="Student", color="coral")
    ax.set_xlabel("Attack Type"); ax.set_ylabel("EER (%)")
    ax.set_title("Per-Attack EER: Teacher vs Student (Eval Set)")
    ax.set_xticks(x); ax.set_xticklabels(merged["Attack"], rotation=45, ha="right")
    ax.legend(); ax.grid(True, alpha=0.3, axis="y")

    for bar in b1:
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.3,
                f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=7)
    for bar in b2:
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.3,
                f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=7)

    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / "per_attack_eer.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 5.10 Hardest & Easiest Attacks

In [ ]:
if not s_attacks.empty:
    print("Top 5 HARDEST attacks for Student:")
    print(s_attacks.nlargest(5, "EER (%)").to_string(index=False))
    print("\nTop 5 EASIEST attacks for Student:")
    print(s_attacks.nsmallest(5, "EER (%)").to_string(index=False))

## 5.11 Save All Results

In [ ]:
summary = {
    "teacher": {k: float(teacher_res[k]) for k in ["eer","min_tdcf","accuracy","precision","recall","f1"]},
    "student": {k: float(student_res[k]) for k in ["eer","min_tdcf","accuracy","precision","recall","f1"]},
    "teacher_params": count_params(teacher, True),
    "student_params": count_params(student, True),
    "config": {"max_duration": MAX_DURATION, "target_sr": TARGET_SR, "seed": SEED},
}

with open(str(RESULTS_DIR / "results_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

if not t_attacks.empty:
    t_attacks.to_csv(str(RESULTS_DIR / "teacher_per_attack.csv"), index=False)
if not s_attacks.empty:
    s_attacks.to_csv(str(RESULTS_DIR / "student_per_attack.csv"), index=False)

In [ ]:
print("\n" + "=" * 60)
print("PROJECT COMPLETE!")
print("=" * 60)
print(f"""
Files saved to Google Drive ({RESULTS_DIR}):
  results_summary.json
  confusion_matrices.png
  score_distributions.png
  per_attack_eer.png
  teacher_per_attack.csv
  student_per_attack.csv

Checkpoints ({CHECKPOINT_DIR}):
  teacher_best.pt
  student_best.pt
""")